# D_Cortex Pas 7a — D9 FULL EVAL (Gates 0..9)

## Cum se rulează (zero upload extern)
1. Runtime: **A100 GPU** (Runtime -> Change runtime type -> A100).
2. Runtime -> **Run all**.

Notebook complet self-contained. `code.py` (~750KB, 18k+ linii) este embedat în Cell 3 ca payload base64. Nu trebuie upload de fișiere externe.

Mountează Drive (pentru shadow checkpoint + dataset cache + rezultate), setează `MODE=pas6_full` + `V15_7A_D9_MODE=run`, execută TOT pipeline-ul (v15.1 -> Pas 6 -> Pas 7a D6/D7/D8/D9), salvează JSON și printează verdictele în ordinea: **Gate 0 -> Gate 4 -> Gate 3 -> Gates 5-9 -> Gates 1, 2**.

Output canonic: `/content/drive/MyDrive/dcortex_v2/v15_7a/results/v15_7a_d9_full_eval.json`


In [ ]:
# === CELL 1: setup (Drive mount + env vars) ===
import os, sys, json

from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/dcortex_v2'
os.makedirs(PROJECT_ROOT, exist_ok=True)
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')

os.environ['MODE']                  = 'pas6_full'
os.environ['V15_7A_D9_MODE']        = 'run'
for k in ('D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8'):
    os.environ[f'V15_7A_{k}_MODE'] = 'skip'
os.environ['V15_7A_D9_STRUCT_MODE'] = 'skip'

print('[setup] env vars setate:')
for k in sorted(os.environ):
    if k.startswith('V15_7A_') or k == 'MODE':
        print(f'    {k} = {os.environ[k]}')


In [ ]:
# === CELL 2: embed code.py inline (base64 — single-shot, fără upload extern) ===
import base64
_PAYLOAD_B64 = '''\
...
'''
EMBEDDED_CODE_PY = base64.b64decode(_PAYLOAD_B64.replace('\n', '')).decode('utf-8')
print(f'[embed] code.py decoded: {len(EMBEDDED_CODE_PY):,} chars, {EMBEDDED_CODE_PY.count(chr(10)):,} lines')


In [ ]:
# === CELL 3: execută TOT pipeline-ul ===
print('=' * 78)
print('[D9] EXECUTING FULL PIPELINE (v15.1 -> Pas 6 -> Pas 7a D9)')
print('=' * 78)

PIPELINE_NS = {'__name__': '__main__'}
exec(compile(EMBEDDED_CODE_PY, 'code.py', 'exec'), PIPELINE_NS)


In [ ]:
# === CELL 4: D9 verdict + localization (priority: 0->4->3->5-9->1,2) ===
d9 = PIPELINE_NS.get('d9_result')
gate_descriptions = PIPELINE_NS.get('V15_7A_D9_GATES', {})

if d9 is None:
    print('\n[D9] EROARE: d9_result nu a fost produs.')
    print('    Cauze: V15_7A_D9_MODE nu picked up sau Pas 6 stack neîncărcat.')
    raise SystemExit(1)

print()
print('=' * 78)
print('D9 VERDICTS  (priority: Gate 0 -> Gate 4 -> Gate 3 -> 5,6,7,8,9 -> 1,2)')
print('=' * 78)

priority = ['Gate 0', 'Gate 4', 'Gate 3',
            'Gate 5', 'Gate 6', 'Gate 7', 'Gate 8', 'Gate 9',
            'Gate 1', 'Gate 2']
verdicts = d9['verdicts']
for g in priority:
    ok = verdicts[g]
    desc = gate_descriptions.get(g, '')
    print(f"  [{'PASS' if ok else 'FAIL'}] {g}: {desc}")

print()
print(f"OVERALL: {'PAS 7A SEALED (toate 10 gates verzi)' if d9['overall'] else 'BLOCKED'}")

V15_7A_RESULTS_DIR = os.path.join(PROJECT_ROOT, 'v15_7a', 'results')
os.makedirs(V15_7A_RESULTS_DIR, exist_ok=True)
d9_path = os.path.join(V15_7A_RESULTS_DIR, 'v15_7a_d9_full_eval.json')
if not os.path.isfile(d9_path):
    json_safe = PIPELINE_NS.get('_v15_7a_json_safe', lambda x: x)
    with open(d9_path, 'w') as f:
        json.dump(json_safe(d9), f, indent=2, default=str)
print(f'\n[D9] artifact: {d9_path}')

def _hr(label):
    print(); print('-' * 78); print(label); print('-' * 78)

if not verdicts['Gate 0']:
    _hr('Gate 0 LOCALIZATION — trusted snapshot delta')
    sb = d9['phase_a']['trusted_snapshot_before']
    sa = d9['phase_a']['trusted_snapshot_after']
    any_diff = False
    for k in sb:
        b, a = sb[k], sa.get(k)
        if b != a:
            any_diff = True
            print(f'  CHANGED  {k:30s}  before={b}  after={a}')
    if not any_diff:
        print('  Snapshot keys identice numeric — verifică tuple-level pe S5/S6 manual.')

if not verdicts['Gate 4']:
    _hr('Gate 4 LOCALIZATION — sequences cu false_retrograde > 0')
    for fam, trials in d9['phase_b']['per_l_family'].items():
        bad = [t for t in trials if t['metrics']['false_retrograde'] > 0]
        if not bad: continue
        print(f'\n  {fam}: {len(bad)}/{len(trials)} trials')
        for t in bad[:5]:
            print(f"    seq={t['sequence_id']}")
            print(f"      expected_ops = {t['expected_ops']}")
            print(f"      actual: RECON={t['metrics']['actual_RECONCILE']} "
                  f"PRUNE={t['metrics']['actual_PRUNE']} "
                  f"RETRO={t['metrics']['actual_RETROGRADE']} "
                  f"PROMO={t['metrics']['actual_PROMOTE']}")
            print(f"      final_committed   = {t['final_committed']}")
            print(f"      expected_committed= {t['expected_committed']}")

if not verdicts['Gate 3']:
    _hr('Gate 3 LOCALIZATION — sequences cu false_promote > 0')
    for fam, trials in d9['phase_b']['per_l_family'].items():
        bad = [t for t in trials if t['metrics']['false_promote'] > 0]
        if not bad: continue
        print(f'\n  {fam}: {len(bad)}/{len(trials)} trials')
        for t in bad[:5]:
            print(f"    seq={t['sequence_id']}")
            print(f"      expected_ops = {t['expected_ops']}")
            print(f"      actual_PROMOTE = {t['metrics']['actual_PROMOTE']}")
            print(f"      final_committed = {t['final_committed']}")

if not verdicts['Gate 5']:
    _hr(f"Gate 5 — L1 promote_rate = {d9['phase_b']['l1_promote_rate']:.3f} (need >= 0.95)")
if not verdicts['Gate 6']:
    _hr(f"Gate 6 — L2 retrograde_rate = {d9['phase_b']['l2_retrograde_rate']:.3f} (need >= 0.90)")
if not verdicts['Gate 7']:
    _hr(f"Gate 7 — L3 false_retrograde count = {d9['phase_b']['l3_false_retrograde_count']} (need 0)")
if not verdicts['Gate 8']:
    _hr(f"Gate 8 — L4 promote_count = {d9['phase_b']['l4_promote_count']} (need 0)")
if not verdicts['Gate 9']:
    _hr('Gate 9 — L5 prune_per_trial:')
    print(f"  {d9['phase_b']['l5_prune_per_trial']}")
    print('  (need >= 1 per trial)')

if not verdicts['Gate 1']:
    _hr('Gate 1 LOCALIZATION — wrong_commit per family')
    for fname, r in d9['phase_a']['family_results'].items():
        wc = r.get('wrong_commit_rate', 0)
        marker = '  X' if wc > 0.02 else '   '
        print(f'  {marker} {fname}: wrong_commit={wc:.4f}')

if not verdicts['Gate 2']:
    _hr('Gate 2 LOCALIZATION — F2 safe_resolution')
    f2 = d9['phase_a']['family_results'].get('F2_multiword_entities', {})
    safe = f2.get('commit_correct_rate', 0) + f2.get('provisional_correct_rate', 0)
    print(f"  F2 safe_resolution = {safe:.4f} (need >= 0.95)")

print()
print('=' * 78)
print('D9 RUN COMPLETE — paste output (de la "D9 VERDICTS" în jos) înapoi')
print('=' * 78)
